# Remaining Seed-42 Experiments -- Kaggle Notebooks

Extends the previous `core_seed42` run (Experiments 0, A, B, C, D already
trained and saved as that earlier notebook's own Kaggle output) with
everything still outstanding at seed 42:

| Missing piece | What it is |
|---|---|
| `exp_0b_plain_deep18_no_skip_shared_adapters_learned_balance` | Depth/width-matched **no-skip-connection** control -- the ablation that actually isolates whether residual connections help, holding depth/width fixed. |
| `exp_0c_custom_resnet18_no_zero_init_shared_adapters_learned_balance` | Same as Experiment D but `zero_init_residual=false` -- isolates the effect of zero-initializing each residual branch. |
| `exp_e_parametric_vs_knn` | Not a training run -- builds a k-NN index on Experiment D's embedding space and re-evaluates with `--compare-knn`, so the parametric heads can be compared against a non-parametric baseline. |
| `exp_f_pretrained_vs_scratch` (optional, **off by default**) | SimCLR self-supervised pretraining (`scripts/pretrain.py`) followed by supervised fine-tuning. Heavy; toggle `RUN_EXP_F` below if you want it. |
| **VOLO-D1 face-only** (`docs/transfer_learning.md`) | ImageNet-pretrained VOLO-D1 transformer backbone, face-only crop, fed into the *same* adapters/heads/loss-balancing as the core suite. Supplementary Table B, never mixed into the core suite's Table A. |

**Design.** This notebook reuses this project's own proven orchestration
functions (`train_one_experiment`, `calibrate_one_experiment`,
`build_knn_one_experiment`, `evaluate_one_experiment`,
`run_experiment_pipeline`, `mirror_to_run_level`, `generate_final_results_report`,
the VOLO `TransferTrainer`/`PersistentArtifactManager` bootstrap) --
copied verbatim from `notebooks/train_evaluate_kaggle.ipynb`, not
reimplemented. It resumes the **same `RUN_ID`** as the original run.

This notebook never mounts Google Drive and never imports `google.colab`.
Instead, it restores Experiments 0/A/B/C/D either from an attached Kaggle
input dataset holding the previous run's own saved output (recommended --
see section 3), or from an optional, secure Google Drive backup (Kaggle
Secrets only, credentials never written to disk). Everything produced in
*this* session is saved directly under `/kaggle/working`, which becomes
this notebook version's own output once saved -- so the final comparison
tables/plots cover all nine core experiments plus the two supplementary
Table-B rows together, and this session's own output can in turn be
attached to a later follow-up session the same way the previous run's was.

Every already-complete stage (checkpoint / calibration / k-NN index /
metrics already on disk) is skipped automatically -- rerunning this
notebook after an interruption is always safe.


## 1. User configuration


In [ ]:
# ============================================================
# USER CONFIGURATION -- edit this cell, then run the notebook top to bottom.
# ============================================================

REPO_URL = "https://github.com/adischwartz15/AgeGender.git"
REPO_BRANCH = "main"

# The RUN_ID of the original seed=42 run whose output already holds
# Experiments 0/A/B/C/D (printed by that notebook run, and the name of the
# run directory inside its saved Kaggle output). Change this if your
# original run used a different RUN_ID.
PREVIOUS_RUN_ID = "2026-07-13_1210_core_seed42"

SEED = 42
MAX_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 12
MAX_BATCHES_PER_EPOCH = None
FORCE_RERUN = False  # True would retrain everything, including 0/A/B/C/D -- leave False.
ALLOW_TEST_FAILURES = False

# Same training-quality defaults as the original notebook -- identical
# across every experiment, so results stay comparable.
DIFFERENTIAL_LR_ENABLED = True
BACKBONE_LR_MULTIPLIER = 0.1
ADAPTER_BOTTLENECK_DIM = 256
LOSS_BALANCING_WARMUP_EPOCHS = 3

# --- Where to restore Experiments 0/A/B/C/D from --------------------------
# Recommended: save the ORIGINAL notebook's output as a version, add that
# version's output as an input dataset to THIS session (Add Data > Your
# Datasets/Notebooks), and point this at the PREVIOUS_RUN_ID directory
# inside it, e.g.
# "/kaggle/input/agegender-core-seed42-output/2026-07-13_1210_core_seed42".
PREVIOUS_RUN_KAGGLE_INPUT_DIR = None

# Optional, secure Google Drive backup/restore of the run directory (see
# src/utils/kaggle_drive_backup.py) -- used only as a fallback restore
# source, or if PREVIOUS_RUN_KAGGLE_INPUT_DIR above is not set. Requires
# GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON + GOOGLE_DRIVE_FOLDER_ID in Kaggle
# Secrets (Add-ons > Secrets) -- credentials are never written to disk,
# printed, or logged.
ENABLE_KAGGLE_DRIVE_BACKUP = False
KAGGLE_RESTORE_SOURCE = "auto"  # "attached_dataset" | "drive" | "working_directory" | "auto"

# Path to an attached Kaggle dataset holding the raw images, e.g.
# "/kaggle/input/utkface-new". Only needed if the dataset itself isn't
# already sitting under data/raw from earlier in this same session.
KAGGLE_INPUT_DATASET_DIR = None

# Optional: only needed if downloading the dataset via the Kaggle API instead.
KAGGLE_DATASET_SLUG = None      # e.g. "jangedoo/utkface-new"

# --- Which missing pieces to run -----------------------------------------
RUN_EXP_0B = True
RUN_EXP_0C = True
RUN_EXP_E_KNN = True
# Off by default -- needs scripts/pretrain.py (SimCLR self-supervised
# pretraining) first, a genuinely separate, heavy additional training
# phase, and exp_f is documented as optional/auto-skipping upstream too.
RUN_EXP_F = False
RUN_NONPARAMETRIC_BASELINES = True   # CPU-only, cheap, real value for "parametric vs non-parametric" comparisons
RUN_TRANSFER_LEARNING_EXTENSION = True   # the VOLO-D1 face-only experiment
TRANSFER_SEEDS = [42]  # only seed 42 is outstanding; set e.g. [42, 123, 2026] to also (re)run the others

print(f"PREVIOUS_RUN_ID = {PREVIOUS_RUN_ID}")
print(f"PREVIOUS_RUN_KAGGLE_INPUT_DIR = {PREVIOUS_RUN_KAGGLE_INPUT_DIR}")
print(f"KAGGLE_RESTORE_SOURCE = {KAGGLE_RESTORE_SOURCE}  ENABLE_KAGGLE_DRIVE_BACKUP = {ENABLE_KAGGLE_DRIVE_BACKUP}")
print(f"Missing-piece toggles: exp_0b={RUN_EXP_0B} exp_0c={RUN_EXP_0C} exp_e_knn={RUN_EXP_E_KNN} "
      f"exp_f={RUN_EXP_F} nonparametric_baselines={RUN_NONPARAMETRIC_BASELINES} "
      f"volo_transfer_learning={RUN_TRANSFER_LEARNING_EXTENSION}")


In [ ]:
# ============================================================
# Helper library -- shared plumbing used by every phase below.
# None of this reimplements model/dataset/training/evaluation logic; it only
# runs the repository's own scripts, and manages files/manifests around them.
# ============================================================

import datetime
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from IPython.display import Image, Markdown, display


def run_command(cmd, cwd=None, log_path=None, check=True, env=None):
    """Run a subprocess, streaming output live and optionally capturing it to a log file."""
    printable = cmd if isinstance(cmd, str) else " ".join(str(c) for c in cmd)
    print(f"$ {printable}")
    full_env = {**os.environ, **(env or {})}
    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, shell=isinstance(cmd, str), env=full_env,
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    proc.wait()
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text("".join(lines), encoding="utf-8")
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {proc.returncode}): {printable}"
            + (f"\nSee log: {log_path}" if log_path else "")
        )
    return proc.returncode, "".join(lines)


def write_manifest(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(data, fh, indent=2, default=str)
    return path


def load_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_copy2(src, dst):
    """shutil.copy2, but skips (with a message) instead of raising when src and dst resolve to the same file.

    This happens on a resumed run whose RUN_DIR already points directly at
    the persistent output/run directory (e.g. a run resumed from a
    previously-restored RUN_DIR): the log/checkpoint/metrics files under
    RUN_DIR are already being written to their final destination, so a
    later "sync to persistent storage" copy step would otherwise raise
    shutil.SameFileError trying to copy a file onto itself.
    """
    src, dst = Path(src), Path(dst)
    if src.resolve() == dst.resolve():
        print(f"Skipping copy: source and destination are the same file ({dst})")
        return dst
    return shutil.copy2(src, dst)


def copy_tree_merge(src, dst):
    """Copy every file under src into dst (creating dst), without touching unrelated existing dst content."""
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return []
    dst.mkdir(parents=True, exist_ok=True)
    copied = []
    for path in src.rglob("*"):
        if path.is_file():
            target = dst / path.relative_to(src)
            target.parent.mkdir(parents=True, exist_ok=True)
            safe_copy2(path, target)
            copied.append(target)
    return copied


def move_tree_clear(src, dst):
    """Move every file under src into dst, then remove the now-empty src.

    Used for the couple of scripts (generate_gradcam.py, run_robustness.py)
    whose output directory is not configurable per run, so consecutive
    experiments would otherwise collide.
    """
    copied = copy_tree_merge(src, dst)
    if Path(src).exists():
        shutil.rmtree(src)
    return copied


def flatten_overrides(obj, prefix=""):
    """Flatten a nested dict into a flat list of ["--set", "a.b.c=value", ...] CLI args.

    Mirrors src/utils/config.py's own --set parsing (parse_cli_overrides /
    _coerce_scalar) exactly, so every value round-trips the same way it
    would from a hand-typed CLI override.
    """
    args = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            new_prefix = f"{prefix}.{key}" if prefix else key
            args.extend(flatten_overrides(value, new_prefix))
    else:
        value = obj
        if value is None:
            value = "null"
        elif isinstance(value, bool):
            value = "true" if value else "false"
        args += ["--set", f"{prefix}={value}"]
    return args


def display_image(path, caption=None):
    path = Path(path)
    if not path.exists():
        print(f"(plot not available: {path})")
        return
    if caption:
        display(Markdown(f"**{caption}**"))
    display(Image(filename=str(path)))


def artifact_ready(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0


def validate_required_artifacts(paths):
    missing = [str(p) for p in paths if not artifact_ready(p)]
    if missing:
        raise RuntimeError("Required artifact(s) missing or empty:\n  " + "\n  ".join(missing))
    return True


print("Helper library loaded.")


## 2. Runtime, GPU, repository, and dependencies


In [ ]:
# ============================================================
# Detect Kaggle
# ============================================================

IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/input").exists()
print(f"Running in Kaggle: {IN_KAGGLE}")
if not IN_KAGGLE:
    print(
        "This notebook is designed for Kaggle Notebooks. It may still run "
        "elsewhere, but paths under /kaggle are Kaggle-specific; consider "
        "the Colab version of this notebook or a local checkout instead."
    )


In [ ]:
# ============================================================
# Runtime and GPU verification
# ============================================================

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU memory (GB): {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
    else:
        print(
            "No GPU detected -- training will run on CPU and will be substantially "
            "slower. Enable a GPU accelerator in this platform's runtime settings "
            "before running the training cells below."
        )
except ImportError:
    print("PyTorch not yet installed -- run the dependency installation cell below first.")


In [ ]:
# ============================================================
# Repository setup
# ============================================================

REPO_DIR = Path("/kaggle/working/AgeGender")

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print(f"Repository already present at {REPO_DIR}; fetching latest {REPO_BRANCH}...")
    run_command(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run_command(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
    run_command(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_DIR)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_command(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")


In [ ]:
# ============================================================
# Dependency installation
# ============================================================

run_command(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
)

# Editable install is a convenience for `import src...` directly from notebook
# cells; every script under scripts/ already inserts the repo root onto its
# own sys.path and does not depend on this succeeding. pyproject.toml also
# currently declares requires-python >=3.11, so this is skipped gracefully
# (not treated as fatal) on older interpreters.
try:
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)
    print("Editable install succeeded.")
except Exception as exc:
    print(f"Editable install skipped ({exc}); continuing with sys.path only (this is expected and fine).")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch  # noqa: E402  (re-import after installation, in case this is a fresh session)

print("Dependencies installed.")


## 3. Run directory, restore from a previous run, and environment manifest

Recreates the **same run directory** (`RUN_ID = PREVIOUS_RUN_ID`) locally
under `/kaggle/working`, and restores everything already saved for
Experiments 0/A/B/C/D -- both into the run's own `experiments/`/flat mirror
directories (so the restart-safety checks in the training helpers see them
as already complete) and into the repository's own default
`checkpoints/`/`outputs/` locations (so `scripts/run_transfer_learning.py`'s
baseline lookup, which reads those default paths, finds Experiment D
without retraining it). This notebook never mounts Google Drive -- see
`KAGGLE_RESTORE_SOURCE` in section 1 for the restore sources it does use.


In [ ]:
# ============================================================
# Run ID and run directory -- always resumes PREVIOUS_RUN_ID, then restores
# its previously-saved content into this fresh session before anything else
# reads it. Never imports google.colab and never calls
# google.colab.drive.mount(...) -- see KAGGLE_RESTORE_SOURCE in section 1.
# ============================================================

WORKSPACE_DIR = Path("/kaggle/working/agegender_runs")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = PREVIOUS_RUN_ID
RUN_DIR = WORKSPACE_DIR / RUN_ID

RUN_SUBDIRS = [
    "config", "logs", "tests", "data_quality", "checkpoints", "metrics", "plots",
    "calibration", "gradcam", "robustness", "knn", "reports", "manifests",
    "archives", "experiments",
]
for _sub in RUN_SUBDIRS:
    (RUN_DIR / _sub).mkdir(parents=True, exist_ok=True)

_restored_anything = False

if KAGGLE_RESTORE_SOURCE in ("attached_dataset", "auto") and PREVIOUS_RUN_KAGGLE_INPUT_DIR:
    _prev_run_dir = Path(PREVIOUS_RUN_KAGGLE_INPUT_DIR)
    if _prev_run_dir.exists():
        for _sub in RUN_SUBDIRS:
            _restored = copy_tree_merge(_prev_run_dir / _sub, RUN_DIR / _sub)
            if _restored:
                print(f"[attached_dataset] restored {len(_restored)} file(s) into {RUN_DIR / _sub}.")
                _restored_anything = True
    else:
        print(f"WARNING: PREVIOUS_RUN_KAGGLE_INPUT_DIR={_prev_run_dir} does not exist -- nothing restored from it.")

if KAGGLE_RESTORE_SOURCE in ("drive", "auto") and not _restored_anything and ENABLE_KAGGLE_DRIVE_BACKUP:
    from src.utils.kaggle_drive_backup import download_file, is_configured as _drive_configured

    if _drive_configured():
        _prev_archive_zip = RUN_DIR.parent / f"{PREVIOUS_RUN_ID}_archive_restored.zip"
        if download_file(_prev_archive_zip, drive_filename=f"{PREVIOUS_RUN_ID}_archive.zip"):
            with zipfile.ZipFile(_prev_archive_zip) as _zf:
                _zf.extractall(RUN_DIR)
            print(f"[drive] restored run directory from {_prev_archive_zip.name}.")
            _restored_anything = True
        else:
            print("[drive] no previous-run archive restored (missing backup, or network unavailable).")
    else:
        print(
            "[drive] ENABLE_KAGGLE_DRIVE_BACKUP=True but GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON/"
            "GOOGLE_DRIVE_FOLDER_ID Kaggle Secrets are not both set -- Drive restore skipped."
        )

if not _restored_anything:
    if KAGGLE_RESTORE_SOURCE == "working_directory":
        print(
            "KAGGLE_RESTORE_SOURCE='working_directory' -- using whatever is already under "
            f"{RUN_DIR} in this session (no cross-session restore attempted)."
        )
    else:
        print(
            f"WARNING: nothing restored for RUN_ID={RUN_ID} -- Experiments 0/A/B/C/D will be "
            "retrained from scratch instead of reused. Set PREVIOUS_RUN_KAGGLE_INPUT_DIR (recommended) "
            "or ENABLE_KAGGLE_DRIVE_BACKUP if this is unexpected."
        )

# Also materialize the flat mirror into the repo's own DEFAULT checkpoint/
# output locations (configs/default.yaml: paths.checkpoint_dir=./checkpoints,
# paths.output_dir=./outputs) -- scripts/run_transfer_learning.py's baseline
# lookup and scripts/build_knn_index.py/evaluate.py's default invocations
# read from there, not from RUN_DIR.
_default_dir_map = {
    "checkpoints": REPO_DIR / "checkpoints",
    "metrics": REPO_DIR / "outputs" / "metrics",
    "plots": REPO_DIR / "outputs" / "plots",
    "calibration": REPO_DIR / "outputs" / "calibration",
    "knn": REPO_DIR / "outputs" / "knn",
    "reports": REPO_DIR / "outputs" / "reports",
    "robustness": REPO_DIR / "outputs" / "robustness",
    "gradcam": REPO_DIR / "outputs" / "gradcam",
    "data_quality": REPO_DIR / "outputs" / "data_quality",
}
for _run_sub, _default_dest in _default_dir_map.items():
    _n = copy_tree_merge(RUN_DIR / _run_sub, _default_dest)
    if _n:
        print(f"Mirrored {len(_n)} file(s) into default location {_default_dest}")

print(f"RUN_ID  = {RUN_ID}")
print(f"RUN_DIR = {RUN_DIR}")


In [ ]:
# ============================================================
# Environment manifest for this supplementary session (kept separate from
# the original run's manifests/environment.json so neither overwrites the
# other -- both live under the same RUN_DIR/manifests/).
# ============================================================

def _get_git_info(repo_dir):
    try:
        sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
        branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=repo_dir, text=True).strip()
        status = subprocess.check_output(["git", "status", "--short"], cwd=repo_dir, text=True)
    except Exception as exc:
        sha, branch, status = f"unavailable ({exc})", "unavailable", ""
    return sha, branch, status


git_sha, git_branch, git_status = _get_git_info(REPO_DIR)

gpu_info = {}
if torch.cuda.is_available():
    gpu_info = {
        "name": torch.cuda.get_device_name(0),
        "total_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
        "cuda_version": torch.version.cuda,
    }

environment_manifest = {
    "run_id": RUN_ID,
    "session": "remaining_experiments_seed42",
    "git_commit_sha": git_sha,
    "git_branch": git_branch,
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": gpu_info,
    "platform": platform.platform(),
    "generated_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
}
write_manifest(RUN_DIR / "manifests" / "environment_remaining_seed42.json", environment_manifest)
print(json.dumps(environment_manifest, indent=2))


## 4. Dataset and locked split

`data/` is gitignored and not part of the previous run's saved output, so
the raw images need to be available again in this fresh session -- either
through an attached Kaggle input dataset (`KAGGLE_INPUT_DATASET_DIR`,
recommended, read-only, not copied) or a Kaggle API download
(`KAGGLE_DATASET_SLUG`). The split itself
(`data_quality/full_metadata_with_splits.csv`) **was** restored in section 3
above (when a restore source was configured), so `scripts/prepare_data.py`
below reuses the exact same locked split as the original run rather than
regenerating a new one -- required for the new experiments' results to be
comparable to the existing ones.


In [ ]:
# ============================================================
# Dataset setup
# This notebook never mounts Google Drive -- Kaggle datasets are attached
# read-only under /kaggle/input/... via the notebook's 'Add Data' UI, or
# downloaded via the Kaggle API if KAGGLE_DATASET_SLUG is set instead.
# ============================================================

prepare_dataset_override = []
if KAGGLE_INPUT_DATASET_DIR:
    print(f"Using attached Kaggle input dataset at {KAGGLE_INPUT_DATASET_DIR} (read-only, not copied).")
    prepare_dataset_override = ["--set", f"dataset.image_root={KAGGLE_INPUT_DATASET_DIR}"]
elif KAGGLE_DATASET_SLUG:
    os.environ["KAGGLE_DATASET_SLUG"] = KAGGLE_DATASET_SLUG
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        raise RuntimeError(
            "KAGGLE_DATASET_SLUG is set but KAGGLE_USERNAME/KAGGLE_KEY are not. "
            "Add them as Kaggle 'Secrets' (Add-ons -> Secrets) and load them "
            "into os.environ before this cell, or attach the dataset directly "
            "via 'Add Data' and set KAGGLE_INPUT_DATASET_DIR instead."
        )
    run_command(
        [sys.executable, "-u", "scripts/download_kaggle_data.py"],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "download_data.log",
    )
else:
    raise RuntimeError(
        "No dataset source configured. Set KAGGLE_INPUT_DATASET_DIR to an "
        "attached dataset path (e.g. '/kaggle/input/utkface-new') or "
        "KAGGLE_DATASET_SLUG to download via the Kaggle API. Stopping rather "
        "than guessing a path."
    )


In [ ]:
# ============================================================
# Data validation and deterministic split preparation
# ============================================================

split_csv_path = RUN_DIR / "data_quality" / "full_metadata_with_splits.csv"
prepare_overrides = [
    "--set", f"paths.splits_dir={(RUN_DIR / 'data_quality').as_posix()}",
    "--set", f"validation.report_dir={(RUN_DIR / 'data_quality').as_posix()}",
] + prepare_dataset_override

if split_csv_path.exists() and not FORCE_RERUN:
    print(f"Reusing existing prepared split at {split_csv_path} (FORCE_RERUN=False).")
else:
    run_command(
        [sys.executable, "-u", "scripts/prepare_data.py"] + prepare_overrides,
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "prepare_data.log",
    )

quality_report = load_json(RUN_DIR / "data_quality" / "data_quality_report.json")
split_hash = sha256_file(split_csv_path)

print("Data quality report:")
print(json.dumps(quality_report, indent=2))
print(f"\nSplit file SHA-256: {split_hash}")

write_manifest(
    RUN_DIR / "manifests" / "data_manifest.json",
    {**quality_report, "split_file_sha256": split_hash, "split_file": str(split_csv_path)},
)


In [ ]:
# ============================================================
# Materialize the locked split at the DEFAULT location too
# (REPO_DIR/data/splits/full_metadata_with_splits.csv, i.e.
# configs/default.yaml: paths.splits_dir=./data/splits). The cell above
# only writes it under RUN_DIR/data_quality/ via an explicit --set
# override -- fine for scripts this notebook already overrides
# paths.splits_dir for (train_one_experiment does), but scripts/pretrain.py
# and scripts/run_transfer_learning.py fall back to (or, in a couple of
# spots, hardcode) the plain default path, so without this copy they fail
# with "No prepared split found" even though section 4 above already
# prepared one.
# ============================================================

_default_splits_dir = REPO_DIR / "data" / "splits"
_default_splits_dir.mkdir(parents=True, exist_ok=True)
_default_split_csv = _default_splits_dir / "full_metadata_with_splits.csv"
if split_csv_path.exists():
    safe_copy2(split_csv_path, _default_split_csv)
    print(f"Materialized locked split at default location: {_default_split_csv}")
else:
    print(f"WARNING: {split_csv_path} not found -- section 4 above must have failed.")


## 5. Automated tests


In [ ]:
# ============================================================
# Automated tests (must pass, or ALLOW_TEST_FAILURES=True, before training)
# ============================================================

pytest_log = RUN_DIR / "tests" / "pytest.log"
junit_path = RUN_DIR / "tests" / "pytest_junit.xml"

returncode, _ = run_command(
    [sys.executable, "-m", "pytest", "-q", f"--junitxml={junit_path}"],
    cwd=REPO_DIR, log_path=pytest_log, check=False,
)
tests_passed = returncode == 0

print(f"\nPython tests: {'PASS' if tests_passed else 'FAIL'}")
print("Frontend build: not checked in this supplementary notebook (see the original run's notebook session for that).")

if not tests_passed and not ALLOW_TEST_FAILURES:
    raise RuntimeError(
        "Automated tests failed and ALLOW_TEST_FAILURES=False. Fix the failures "
        f"(see {pytest_log}) before running expensive training, or set "
        "ALLOW_TEST_FAILURES=True in the configuration cell to proceed anyway "
        "(not recommended before a reported research run)."
    )


## 6. Experiment configuration and training helpers


In [ ]:
# ============================================================
# Load configs/experiments.yaml and confirm every experiment this notebook
# needs actually exists (fails loudly, never substitutes a different model).
# ============================================================

import yaml

CNN_EXPERIMENT = "exp_0_simple_cnn_shared_adapters_learned_balance"
PLAIN_DEEP18_EXPERIMENT = "exp_0b_plain_deep18_no_skip_shared_adapters_learned_balance"
NO_ZERO_INIT_EXPERIMENT = "exp_0c_custom_resnet18_no_zero_init_shared_adapters_learned_balance"
SEPARATE_EXPERIMENT = "exp_a_separate"
SHARED_NO_ADAPTERS_EXPERIMENT = "exp_b_shared_no_adapters"
SHARED_ADAPTERS_EXPERIMENT = "exp_c_shared_adapters"
RESNET_EXPERIMENT = "exp_d_shared_adapters_learned_balance"  # the from-scratch baseline

experiments_cfg = yaml.safe_load((REPO_DIR / "configs" / "experiments.yaml").read_text(encoding="utf-8"))["experiments"]

# The full core suite -- already-complete ones (restored in section 3 above)
# are skipped automatically by the restart-safety checks below; only the
# genuinely missing ones (0b, 0c, per the toggles above) actually train.
ALL_CORE_EXPERIMENTS = [
    CNN_EXPERIMENT, PLAIN_DEEP18_EXPERIMENT, NO_ZERO_INIT_EXPERIMENT,
    SEPARATE_EXPERIMENT, SHARED_NO_ADAPTERS_EXPERIMENT, SHARED_ADAPTERS_EXPERIMENT,
    RESNET_EXPERIMENT,
]
selected_experiments = [CNN_EXPERIMENT, SEPARATE_EXPERIMENT, SHARED_NO_ADAPTERS_EXPERIMENT,
                         SHARED_ADAPTERS_EXPERIMENT, RESNET_EXPERIMENT]
if RUN_EXP_0B:
    selected_experiments.append(PLAIN_DEEP18_EXPERIMENT)
if RUN_EXP_0C:
    selected_experiments.append(NO_ZERO_INIT_EXPERIMENT)

missing = [name for name in selected_experiments if name not in experiments_cfg]
if missing:
    raise RuntimeError(f"Experiment(s) not found in configs/experiments.yaml: {missing}")

print(f"Will ensure these are complete (already-done ones are skipped, not retrained): {selected_experiments}")


In [ ]:
# ============================================================
# Training helpers (orchestration only -- calls scripts/train.py,
# scripts/calibrate.py, scripts/build_knn_index.py, scripts/evaluate.py)
#
# Each stage (train / calibrate / build k-NN index / evaluate) is checked
# and skipped independently, based only on whether *that stage's own*
# artifact already exists. This is what makes the pipeline stage-level
# restart-safe: if evaluation crashes after a successful training run, only
# evaluation reruns next time -- training is never redone just because a
# later stage failed.
# ============================================================

def experiment_paths(run_dir, experiment_name, seed):
    base = run_dir / "experiments" / experiment_name / f"seed_{seed}"
    return {
        "base": base,
        "checkpoint_dir": base / "checkpoints",
        "output_dir": base,
        "calibration_dir": base / "calibration",
        "knn_dir": base / "knn",
        "config_dir": base / "config",
        "log_dir": base / "logs",
    }


def _stage_status(artifact_path, force_rerun):
    exists = artifact_path.exists()
    if exists and not force_rerun:
        return "skipped", f"skipped, found at {artifact_path}"
    if exists and force_rerun:
        return "rerun", f"rerunning (FORCE_RERUN=True), would have reused {artifact_path}"
    return "run", "running, artifact missing"


def print_stage_plan(experiment_name, seed, paths, include_knn):
    """Print the skip/run decision for every stage before executing any of
    them, so a restart's behavior is transparent up front."""
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    calibration_artifact = paths["calibration_dir"] / "conformal_calibration.json"
    knn_index = paths["knn_dir"] / "knn_baseline.pkl"
    metrics_path = paths["output_dir"] / "metrics" / f"{experiment_name}_test_metrics.json"

    print(f"[{experiment_name} seed={seed}] Stage plan:")
    print(f"  - training:   {_stage_status(checkpoint, FORCE_RERUN)[1]}")
    print(f"  - calibration: {_stage_status(calibration_artifact, FORCE_RERUN)[1]}")
    if include_knn:
        print(f"  - k-NN index: {_stage_status(knn_index, FORCE_RERUN)[1]}")
    else:
        print("  - k-NN index: not requested (RUN_KNN=False)")
    print(f"  - evaluation: {_stage_status(metrics_path, FORCE_RERUN)[1]}")


def train_one_experiment(experiment_name, seed):
    spec = experiments_cfg[experiment_name]
    paths = experiment_paths(RUN_DIR, experiment_name, seed)
    for value in paths.values():
        value.mkdir(parents=True, exist_ok=True)

    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    if checkpoint.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] training: skipped, checkpoint found ({checkpoint})")
        return paths

    overrides_args = flatten_overrides(spec.get("overrides", {}))
    notebook_overrides = {
        "seed": seed,
        "training": {
            "seed": seed,
            "warm_up_from_scratch": {"epochs": MAX_EPOCHS},
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "max_train_batches_per_epoch": MAX_BATCHES_PER_EPOCH,
            "max_val_batches_per_epoch": MAX_BATCHES_PER_EPOCH,
            "differential_lr": {"enabled": DIFFERENTIAL_LR_ENABLED, "backbone_lr_multiplier": BACKBONE_LR_MULTIPLIER},
        },
        "model": {
            "adapters": {"bottleneck_dim": ADAPTER_BOTTLENECK_DIM},
            "loss_balancing": {"learned_uncertainty": {"warmup_epochs": LOSS_BALANCING_WARMUP_EPOCHS}},
        },
        "paths": {
            "checkpoint_dir": paths["checkpoint_dir"].as_posix(),
            "output_dir": paths["output_dir"].as_posix(),
            "splits_dir": (RUN_DIR / "data_quality").as_posix(),
        },
        "calibration": {"output_dir": paths["calibration_dir"].as_posix()},
        "knn": {"index_dir": paths["knn_dir"].as_posix()},
    }
    overrides_args += flatten_overrides(notebook_overrides)

    write_manifest(
        paths["config_dir"] / "notebook_overrides.json",
        {"experiment_overrides": spec.get("overrides", {}), "notebook_overrides": notebook_overrides},
    )

    print(
        f"[{experiment_name} seed={seed}] training: "
        f"{'rerunning (FORCE_RERUN=True)' if checkpoint.exists() else 'running, checkpoint missing'} "
        f"(max_epochs={MAX_EPOCHS}, patience={EARLY_STOPPING_PATIENCE})"
    )
    run_command(
        [sys.executable, "-u", "scripts/train.py", "--experiment-name", experiment_name] + overrides_args,
        cwd=REPO_DIR, log_path=paths["log_dir"] / "train.log",
    )
    print(f"[{experiment_name} seed={seed}] training: checkpoint written to {checkpoint}")

    # Every checkpoint embeds the full merged config it was produced with
    # (src/training/checkpointing.py:save_checkpoint) -- extract and save it
    # alongside the notebook-level override diff above, so the *actual*
    # resolved configuration is on disk, not just the requested overrides.
    if checkpoint.exists():
        import torch as _torch

        resolved_config = _torch.load(checkpoint, map_location="cpu")["config"]
        write_manifest(paths["config_dir"] / "resolved_config.json", resolved_config)
    return paths


def calibrate_one_experiment(experiment_name, seed, paths):
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    validate_required_artifacts([checkpoint])

    calibration_artifact = paths["calibration_dir"] / "conformal_calibration.json"
    if calibration_artifact.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] calibration: skipped, artifact found ({calibration_artifact})")
        return

    print(
        f"[{experiment_name} seed={seed}] calibration: "
        f"{'rerunning (FORCE_RERUN=True)' if calibration_artifact.exists() else 'running, artifact missing'}"
    )
    run_command(
        [sys.executable, "-u", "scripts/calibrate.py", "--checkpoint", str(checkpoint)],
        cwd=REPO_DIR, log_path=paths["log_dir"] / "calibrate.log",
    )
    print(f"[{experiment_name} seed={seed}] calibration: artifact written to {calibration_artifact}")


def build_knn_one_experiment(experiment_name, seed, paths):
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    knn_index = paths["knn_dir"] / "knn_baseline.pkl"
    if knn_index.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] k-NN index: skipped, index found ({knn_index})")
        return

    print(
        f"[{experiment_name} seed={seed}] k-NN index: "
        f"{'rerunning (FORCE_RERUN=True)' if knn_index.exists() else 'running, index missing'}"
    )
    run_command(
        [sys.executable, "-u", "scripts/build_knn_index.py", "--checkpoint", str(checkpoint)],
        cwd=REPO_DIR, log_path=paths["log_dir"] / "build_knn_index.log",
    )
    print(f"[{experiment_name} seed={seed}] k-NN index: written to {knn_index}")


def evaluate_one_experiment(experiment_name, seed, paths, include_knn):
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    knn_index = paths["knn_dir"] / "knn_baseline.pkl"
    metrics_path = paths["output_dir"] / "metrics" / f"{experiment_name}_test_metrics.json"

    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] evaluation: skipped, metrics found ({metrics_path})")
        return load_json(metrics_path)

    print(
        f"[{experiment_name} seed={seed}] evaluation: "
        f"{'rerunning (FORCE_RERUN=True)' if metrics_path.exists() else 'running, metrics missing'}"
    )
    eval_args = [
        sys.executable, "-u", "scripts/evaluate.py", "--checkpoint", str(checkpoint),
        "--calibration-dir", str(paths["calibration_dir"]),
        "--output-name", f"{experiment_name}_test_metrics",
    ]
    if include_knn and knn_index.exists():
        eval_args += ["--compare-knn", "--knn-path", str(knn_index)]
    run_command(eval_args, cwd=REPO_DIR, log_path=paths["log_dir"] / "evaluate.log")

    validate_required_artifacts([metrics_path])
    print(f"[{experiment_name} seed={seed}] evaluation: metrics written to {metrics_path}")
    return load_json(metrics_path)


def run_experiment_pipeline(experiment_name, seed, include_knn):
    """Run (or skip, per-stage) train -> calibrate -> [k-NN index] -> evaluate for one experiment/seed."""
    preview_paths = experiment_paths(RUN_DIR, experiment_name, seed)
    for value in preview_paths.values():
        value.mkdir(parents=True, exist_ok=True)
    print_stage_plan(experiment_name, seed, preview_paths, include_knn)

    paths = train_one_experiment(experiment_name, seed)
    calibrate_one_experiment(experiment_name, seed, paths)
    if include_knn:
        build_knn_one_experiment(experiment_name, seed, paths)
    metrics = evaluate_one_experiment(experiment_name, seed, paths, include_knn)
    return paths, metrics


def mirror_to_run_level(paths, flat_name):
    """Copy this experiment/seed's metrics/plots/checkpoints/calibration into the
    flat <RUN_DIR>/{metrics,plots,checkpoints,calibration}/ mirror, renaming
    files from their bare experiment_name prefix to flat_name. flat_name is
    the bare experiment name for the primary seed (so
    src.evaluation.final_report's discovery functions find it directly), or
    f"{experiment_name}_seed{seed}" for additional multi-seed runs (matching
    src.evaluation.comparison's seed-aggregation naming convention).
    """
    experiment_name = paths["base"].parent.name
    for sub in ("metrics", "plots"):
        src_dir = paths["output_dir"] / sub
        if not src_dir.exists():
            continue
        for file_path in src_dir.glob(f"{experiment_name}_*"):
            renamed = flat_name + file_path.name[len(experiment_name):]
            dest = RUN_DIR / sub / renamed
            dest.parent.mkdir(parents=True, exist_ok=True)
            safe_copy2(file_path, dest)
    for checkpoint_path in paths["checkpoint_dir"].glob(f"{experiment_name}_*"):
        renamed = flat_name + checkpoint_path.name[len(experiment_name):]
        dest = RUN_DIR / "checkpoints" / renamed
        dest.parent.mkdir(parents=True, exist_ok=True)
        safe_copy2(checkpoint_path, dest)
    copy_tree_merge(paths["calibration_dir"], RUN_DIR / "calibration" / flat_name)


print("Training helpers ready.")


## 7. Train the missing core experiments (0b, 0c)

Experiments 0/A/B/C/D were already trained in the original run and were
restored in section 3 above -- `train_one_experiment` finds their checkpoint
already on disk and skips training, printing "skipped, checkpoint found"
for each. Only Experiments 0b and 0c (if enabled above) actually train.


In [ ]:
# ============================================================
# Training -- runs every selected experiment at SEED. Already-complete
# ones (0/A/B/C/D, restored in section 3) are skipped automatically.
# ============================================================

trained_results = {}
for name in selected_experiments:
    include_knn = RUN_EXP_E_KNN and name == RESNET_EXPERIMENT
    paths, metrics = run_experiment_pipeline(name, SEED, include_knn=include_knn)
    trained_results[name] = {"paths": paths, "test_metrics": metrics}
    mirror_to_run_level(paths, flat_name=name)

    display_image(RUN_DIR / "plots" / f"{name}_training_curves.png", f"{name}: training curves")

    checkpoint_path = paths["checkpoint_dir"] / f"{name}_best_balanced_score.pt"
    print(f"[{name}] age_mae={metrics.get('age_mae')} gender_accuracy={metrics.get('gender_accuracy')} -- DONE")

print(f"\nCompleted {len(trained_results)}/{len(selected_experiments)} experiment(s): {list(trained_results)}")

# Re-mirror into the repo's own default checkpoint/output locations too
# (see section 3) so downstream scripts using default paths (VOLO baseline
# lookup, the API, etc.) see the newly-trained 0b/0c checkpoints as well.
for _run_sub, _default_dest in _default_dir_map.items():
    copy_tree_merge(RUN_DIR / _run_sub, _default_dest)


## 8. Experiment E -- parametric vs. k-NN baseline

Not a separate training run: builds a k-NN index on Experiment D's
embedding space (`scripts/build_knn_index.py`) and re-evaluates Experiment
D's checkpoint with `--compare-knn` (`scripts/evaluate.py`), written under
its own `exp_e_parametric_vs_knn` name so it appears as its own row/entry
in the comparison tables below, matching `configs/experiments.yaml`'s
`exp_e_parametric_vs_knn: base_experiment: exp_d_shared_adapters_learned_balance`.


In [ ]:
# ============================================================
# Experiment E -- reuses Experiment D's checkpoint (never retrains).
# ============================================================

if RUN_EXP_E_KNN:
    if RESNET_EXPERIMENT not in trained_results:
        raise RuntimeError(f"{RESNET_EXPERIMENT} must complete first -- see section 7 above.")

    exp_e_name = "exp_e_parametric_vs_knn"
    _d_paths = trained_results[RESNET_EXPERIMENT]["paths"]
    _d_checkpoint = _d_paths["checkpoint_dir"] / f"{RESNET_EXPERIMENT}_best_balanced_score.pt"
    _d_knn_index = _d_paths["knn_dir"] / "knn_baseline.pkl"

    build_knn_one_experiment(RESNET_EXPERIMENT, SEED, _d_paths)  # skips if already built

    exp_e_metrics_path = _d_paths["output_dir"] / "metrics" / f"{exp_e_name}_test_metrics.json"
    if exp_e_metrics_path.exists() and not FORCE_RERUN:
        print(f"[exp_e] evaluation: skipped, metrics found ({exp_e_metrics_path})")
        exp_e_metrics = load_json(exp_e_metrics_path)
    else:
        run_command(
            [
                sys.executable, "-u", "scripts/evaluate.py", "--checkpoint", str(_d_checkpoint),
                "--calibration-dir", str(_d_paths["calibration_dir"]),
                "--output-name", f"{exp_e_name}_test_metrics",
                "--compare-knn", "--knn-path", str(_d_knn_index),
            ],
            cwd=REPO_DIR, log_path=_d_paths["log_dir"] / "evaluate_exp_e.log",
        )
        validate_required_artifacts([exp_e_metrics_path])
        exp_e_metrics = load_json(exp_e_metrics_path)

    trained_results[exp_e_name] = {"paths": _d_paths, "test_metrics": exp_e_metrics}
    # mirror_to_run_level globs f"{experiment_name}_*" under the ORIGINAL
    # experiment's output dir -- exp_e's files use exp_e_name as their own
    # prefix (via --output-name above), so copy them in directly instead.
    for _f in (_d_paths["output_dir"] / "metrics").glob(f"{exp_e_name}_*"):
        safe_copy2(_f, RUN_DIR / "metrics" / _f.name)
    for _f in (_d_paths["output_dir"] / "plots").glob(f"{exp_e_name}_*"):
        safe_copy2(_f, RUN_DIR / "plots" / _f.name)

    print(json.dumps({k: v for k, v in exp_e_metrics.items() if not isinstance(v, dict)}, indent=2))
else:
    print("Skipped (RUN_EXP_E_KNN=False).")


## 9. Optional: Experiment F -- pretrained-vs-scratch (SimCLR)

**Off by default.** Requires `scripts/pretrain.py` (a full additional
self-supervised pretraining phase) before the supervised fine-tune even
starts. Set `RUN_EXP_F = True` in section 1 to include it.


In [ ]:
# ============================================================
# Optional: Experiment F. SimCLR self-supervised pretraining, then
# supervised fine-tuning with exp_f_pretrained_vs_scratch's overrides
# (configs/experiments.yaml points it at ./checkpoints/simclr_encoder.pt).
# ============================================================

if RUN_EXP_F:
    exp_f_name = "exp_f_pretrained_vs_scratch"
    simclr_checkpoint = REPO_DIR / "checkpoints" / "simclr_encoder.pt"
    if not simclr_checkpoint.exists() or FORCE_RERUN:
        # pretrain.py only accepts --set key=value overrides (see its --help);
        # there is no --seed flag -- seed is set via the config's top-level
        # "seed" key, same convention as flatten_overrides uses everywhere else.
        # paths.splits_dir must also be overridden -- pretrain.py's default
        # (./data/splits) is empty in this run; the locked split prepared in
        # section 4 above only lives under RUN_DIR/data_quality (same value
        # train_one_experiment already passes for every other experiment).
        run_command(
            [
                sys.executable, "-u", "scripts/pretrain.py",
                "--set", f"seed={SEED}",
                "--set", f"paths.splits_dir={(RUN_DIR / 'data_quality').as_posix()}",
            ],
            cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "pretrain_simclr.log",
        )
    else:
        print(f"Reusing existing SimCLR encoder checkpoint: {simclr_checkpoint}")

    if simclr_checkpoint.exists():
        paths, metrics = run_experiment_pipeline(exp_f_name, SEED, include_knn=False)
        trained_results[exp_f_name] = {"paths": paths, "test_metrics": metrics}
        mirror_to_run_level(paths, flat_name=exp_f_name)
        print(f"[{exp_f_name}] age_mae={metrics.get('age_mae')} gender_accuracy={metrics.get('gender_accuracy')} -- DONE")
    else:
        print(f"No SimCLR checkpoint at {simclr_checkpoint} after pretraining -- skipping exp_f, per its documented auto-skip behavior.")
else:
    print("Skipped (RUN_EXP_F=False).")


## 10. Optional: richer non-parametric baselines (raw/PCA + frozen-backbone k-NN + KDE)

CPU-only, no GPU needed. Broader than Experiment E's single k-NN-on-embedding
comparison -- adds raw-pixel/PCA and frozen-backbone k-NN + KDE baselines.


In [ ]:
# ============================================================
# Non-parametric baselines (raw/PCA + frozen-backbone, k-NN + KDE) --
# reuses scripts/tune_nonparametric_baselines.py (validation-only grid
# search) and scripts/evaluate_nonparametric_baselines.py (test-only final
# evaluation, calibration-split-only conformal calibration for the k-NN
# age intervals). CPU-only; does not require a GPU.
# ============================================================

if RUN_NONPARAMETRIC_BASELINES:
    run_command(
        [sys.executable, "-u", str(REPO_DIR / "scripts" / "tune_nonparametric_baselines.py")],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "nonparametric_tune.log",
    )
    run_command(
        [sys.executable, "-u", str(REPO_DIR / "scripts" / "evaluate_nonparametric_baselines.py")],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "nonparametric_evaluate.log",
    )
    nonparametric_results_path = REPO_DIR / "outputs" / "nonparametric" / "results.csv"
    if nonparametric_results_path.exists():
        import pandas as pd
        nonparametric_results = pd.read_csv(nonparametric_results_path)
        display(nonparametric_results)
        shutil.copy2(nonparametric_results_path, RUN_DIR / "reports" / "nonparametric_baselines.csv")
    else:
        print(
            "Non-parametric baseline results were not produced -- most likely no locked split "
            "found (see section 4 above). See the logs above for details."
        )
else:
    print("Skipped (RUN_NONPARAMETRIC_BASELINES=False).")


## 11. Residual-connection ablation (now that Experiment 0b exists)

`scripts/compare_backbones.py`'s full suite (AURC, paired bootstrap CIs,
tail-error analysis, robustness comparison if available, and a final
explicitly-conditional interpretation) across SimpleCNN vs. PlainDeep18NoSkip
vs. Custom ResNet-18 -- this is the comparison that actually isolates
whether residual connections help, holding depth/width fixed.


In [ ]:
# ============================================================
# Backbone comparison suite -- shells out to scripts/compare_backbones.py,
# never reimplements its analysis here.
# ============================================================

if RUN_EXP_0B and PLAIN_DEEP18_EXPERIMENT in trained_results:
    _backbone_short_names = {
        CNN_EXPERIMENT: "simple_cnn",
        PLAIN_DEEP18_EXPERIMENT: "plain_deep18_no_skip",
        RESNET_EXPERIMENT: "custom_resnet18",
    }
    _checkpoint_args, _calibration_args, _robustness_args = [], [], []
    for _exp_name, _short_name in _backbone_short_names.items():
        if _exp_name not in trained_results:
            continue
        _ckpt = trained_results[_exp_name]["paths"]["checkpoint_dir"] / f"{_exp_name}_best_balanced_score.pt"
        _checkpoint_args += ["--checkpoint", f"{_short_name}={_ckpt}"]
        _calibration_args += ["--calibration-dir", f"{_short_name}={trained_results[_exp_name]['paths']['calibration_dir']}"]
        _robustness_csv_path = RUN_DIR / "robustness" / _exp_name / "robustness_results.csv"
        if _robustness_csv_path.exists():
            _robustness_args += ["--robustness-csv", f"{_short_name}={_robustness_csv_path}"]

    backbone_comparison_dir = RUN_DIR / "reports" / "backbone_comparison"
    run_command(
        [
            sys.executable, "-u", "scripts/compare_backbones.py",
            *_checkpoint_args, *_calibration_args, *_robustness_args,
            "--resnet-name", "custom_resnet18", "--output-dir", str(backbone_comparison_dir),
        ],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "compare_backbones.log",
    )
    print(f"Backbone comparison written to {backbone_comparison_dir}")
else:
    print("Skipped (RUN_EXP_0B=False or exp_0b did not complete).")


## 12. Detailed results report and final summary

Auto-discovers every experiment present under `RUN_DIR` (now including
0b/0c/exp_e alongside the original 0/A/B/C/D) -- reuses
`src.evaluation.final_report.generate_final_results_report` unchanged.


In [ ]:
# ============================================================
# Detailed results report -- ablation table, ranking, uncertainty analysis,
# parameter/latency plots across every experiment found under RUN_DIR.
# ============================================================

from src.evaluation.final_report import generate_final_results_report

detailed_report_path = RUN_DIR / "reports" / "detailed_results_report.md"
detailed_report_md = generate_final_results_report(RUN_DIR, report_dir=detailed_report_path.parent)
detailed_report_path.write_text(detailed_report_md, encoding="utf-8")
print(f"Detailed results report saved to {detailed_report_path}")
display(Markdown(detailed_report_md))


In [ ]:
# ============================================================
# Final summary for this supplementary session (kept separate from the
# original run's reports/final_summary.md).
# ============================================================

_lines = [f"# Remaining Seed-42 Experiments -- Summary (Run {RUN_ID})\n"]
_lines.append("## Platform and hardware\n")
_lines.append(f"- Platform: {platform.platform()}\n- GPU: {gpu_info if gpu_info else 'CPU only'}\n"
              f"- Python: {sys.version.split()[0]}\n- PyTorch: {torch.__version__}\n")
_lines.append("## Git commit\n")
_lines.append(f"- SHA: `{git_sha}`\n- Branch: `{git_branch}`\n")
_lines.append("## Experiments completed in this session\n")
for _name, _result in trained_results.items():
    _scalar = {k: v for k, v in _result["test_metrics"].items() if not isinstance(v, dict)}
    _lines.append(f"### {_name}\n\n```\n{json.dumps(_scalar, indent=2)}\n```\n")
_lines.append("## Artifact index\n")
_lines.append(
    f"- Run directory: `{RUN_DIR}`\n"
    "- Detailed results report: `reports/detailed_results_report.md`\n"
    "- Backbone (residual-connection) comparison: `reports/backbone_comparison/`\n"
    "- Non-parametric baselines: `reports/nonparametric_baselines.csv` (if enabled)\n"
    "- VOLO-D1 Table B: `reports/table_b.csv` (if enabled)\n"
)

remaining_summary_md = "\n".join(_lines)
(RUN_DIR / "reports" / "remaining_experiments_summary.md").write_text(remaining_summary_md, encoding="utf-8")
display(Markdown(remaining_summary_md))


## Supplementary Experiment: ImageNet-Pretrained VOLO-D1 (Face-Only)

Everything above (Experiments 0/0b/0c, A-F) is the project's core, controlled
architecture ablation suite: every model in it is trained **from scratch**,
with identical data, splits, and training conditions, so that differences in
outcome can be attributed to the one design choice each experiment isolates
(shared vs. separate backbones, adapters vs. none, fixed vs. learned loss
balancing, residual vs. non-residual backbones). **Nothing above this cell,
and nothing in the core suite's results, is affected by this section.** This
is a separate, clearly labelled **supplementary** experiment answering a
different question:

> How much practical improvement comes from an externally pretrained modern
> visual backbone, relative to our best from-scratch model
> (`exp_d_shared_adapters_learned_balance`)?

**What this is.** One additional model: an **ImageNet-pretrained** VOLO-D1
backbone (`timm`, `volo_d1_224`) feeding the *same* task-specific adapters
(`src/models/adapters.py`) and the *same* learned homoscedastic-uncertainty
loss balancing (`src/losses/multitask_loss.py`) the core suite already uses
-- neither is forked or reimplemented for this section. Only the backbone
and its own resolved preprocessing (input size, mean/std, interpolation,
read from the pretrained model's own config, not this project's 128px
defaults) differ. See `src/models/pretrained_volo.py::PretrainedVOLOFaceOnlyMultiTask`.

**What this is not.**
- **Not MiVOLO.** MiVOLO additionally requires a body crop, a person
  detector, and face+body cross-attention. This is the *face-only* VOLO
  image classifier. MiVOLO is cited in the README only as the motivation
  for trying a VOLO-family backbone on faces specifically, not as the
  architecture used here.
- **Not a UTKFace- or MiVOLO-trained checkpoint.** Backbone weights come
  only from ImageNet. `model.volo.pretrained_source` is validated at model
  construction time against a closed ImageNet-only allow-list.
- **Not a controlled ablation.** VOLO-D1 differs from the from-scratch
  baseline in initialization, pretraining data, parameter count, input
  resolution, **and** its own optimizer/training schedule all at once --
  any difference in Table B reflects the combined effect of all of these,
  never attributed to "the Transformer-style backbone" alone.

**Persistence and resume.** A Kaggle session can be interrupted or time out
at any point during a multi-hour, multi-seed run. The "Persistent
Transfer-Learning Storage and Resume" subsection below saves every
checkpoint, metric, and manifest under `/kaggle/working` (so they become
this notebook version's output artifacts once saved), tracks which seeds
have already finished with a machine-readable completion marker, and lets a
resumed run skip already-complete seeds and continue an interrupted one
from its latest valid checkpoint -- never silently retraining from scratch
when a compatible, resumable checkpoint already exists. An optional,
secure Google Drive backup (Kaggle Secrets only, credentials never written
to disk) is also available. See `docs/transfer_learning.md`
"Persistent artifacts" and "Kaggle recovery".


In [ ]:
# ============================================================
# Supplementary experiment: dependency install (only if enabled above).
# ============================================================

if RUN_TRANSFER_LEARNING_EXTENSION:
    run_command(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-transfer.txt"],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "transfer_learning_install.log",
    )
    print("timm installed for the supplementary transfer-learning extension.")
else:
    print("RUN_TRANSFER_LEARNING_EXTENSION=False -- skipping the supplementary VOLO experiment entirely.")


### Persistent Transfer-Learning Storage and Resume

Everything below is **idempotent and safe to rerun** -- it does not depend
on any Python variable from a previous, now-ended session. It never imports
`google.colab` (this is a Kaggle-only notebook) and never calls
`google.colab.drive.mount(...)`.

**Storage model.** `/kaggle/working` is this notebook's own persistent
output directory -- everything under `LOCAL_TRANSFER_ROOT` becomes part of
the notebook's saved output automatically when you save a version, so no
separate "sync to Drive" step is required for basic persistence. Two
*additional*, optional recovery paths are supported for when
`/kaggle/working` itself is not available at the start of a session (a
brand-new session does not inherit a previous session's `/kaggle/working`
unless you explicitly attach it):

1. **Attached prior-output dataset** (recommended): save this notebook's
   output as a new version, then add that version's output as an input
   dataset to a *new* session (Add Data > Your Datasets/Notebooks), and set
   `KAGGLE_TRANSFER_LEARNING_INPUT_DATASET_DIR` below to its mounted path
   (e.g. `/kaggle/input/<your-notebook-output-dataset-slug>`). This is a
   plain read-only mounted directory, so
   `PersistentArtifactManager.restore_seed()` handles it exactly like any
   other persistent root -- full checkpoint restore, not just a summary.
2. **Optional secure Google Drive backup** (`ENABLE_KAGGLE_DRIVE_BACKUP`):
   uploads artifacts to a Drive folder using a service account, with
   credentials read *only* from Kaggle Secrets
   (`GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON`, `GOOGLE_DRIVE_FOLDER_ID`) -- never
   committed to Git, never written to disk, never printed or logged. See
   `src/utils/kaggle_drive_backup.py`. If the secrets are missing, this
   prints a clear warning and continues saving to `/kaggle/working` without
   failing the run. Drive-based *restore* in this notebook currently
   recovers the lightweight summary archive only (not full per-epoch
   checkpoints) -- the attached-dataset path above is the recommended way
   to resume with full checkpoints from Drive-independent storage.

`KAGGLE_RESTORE_SOURCE` picks which of the above to prefer:
`"attached_dataset"`, `"drive"`, `"working_directory"` (use whatever is
already under `/kaggle/working` in this session, i.e. no cross-session
restore), or `"auto"` (try the attached dataset first, since it is a full
restore, then Drive).

Note: this section's `KAGGLE_RESTORE_SOURCE`/`ENABLE_KAGGLE_DRIVE_BACKUP`
below are scoped to the VOLO-D1 seeds only -- independent of the whole-run
restore configured in section 1/3 above, since VOLO checkpoints are kept
under their own `LOCAL_TRANSFER_ROOT`, separate from `RUN_DIR`.


In [ ]:
# ============================================================
# Persistent Transfer-Learning Storage and Resume -- bootstrap cell.
# Self-contained: reconstructs every path/config this section needs, so
# rerunning environment setup + repository checkout + dependency
# installation + this cell is enough to resume an interrupted run. Safe to
# rerun any number of times. Never imports google.colab.
# ============================================================

PLATFORM = "kaggle"
# TRANSFER_SEEDS is set in the user-configuration cell above (default [42] -- only seed 42 is outstanding)
AUTO_RESUME = True
SKIP_COMPLETED = True
SINGLE_SEED = None  # set e.g. 123 to use the optional single-seed cell further down instead of the full run

# Optional: path to a *previous run's saved output*, attached as an input
# dataset to this session (see the markdown above) -- e.g.
# "/kaggle/input/agegender-transfer-learning-output". None disables this
# restore source.
KAGGLE_TRANSFER_LEARNING_INPUT_DATASET_DIR = None

# Optional, secure Google Drive backup (see src/utils/kaggle_drive_backup.py).
# Requires GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON + GOOGLE_DRIVE_FOLDER_ID in
# Kaggle Secrets (Add-ons > Secrets) -- the target Drive folder must be
# shared with the service account's email address. Independent of
# ENABLE_KAGGLE_DRIVE_BACKUP in section 1 (that one covers the whole-run
# restore; this one covers VOLO-D1 seeds specifically).
ENABLE_KAGGLE_DRIVE_BACKUP = False
KAGGLE_RESTORE_SOURCE = "auto"  # "drive" | "attached_dataset" | "working_directory" | "auto"

LOCAL_TRANSFER_ROOT = Path("/kaggle/working/AgeGender/transfer_learning")
LOCAL_TRANSFER_ROOT.mkdir(parents=True, exist_ok=True)
PERSISTENT_TRANSFER_ROOT = (
    Path(KAGGLE_TRANSFER_LEARNING_INPUT_DATASET_DIR) / "transfer_learning"
    if KAGGLE_TRANSFER_LEARNING_INPUT_DATASET_DIR else None
)

if RUN_TRANSFER_LEARNING_EXTENSION:
    if str(REPO_DIR / "scripts") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "scripts"))
    from src.training.persistent_artifacts import PersistentArtifactManager, format_status_line, seed_status_report
    from src.utils.io import file_sha256

    if ENABLE_KAGGLE_DRIVE_BACKUP:
        run_command(
            [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle-drive.txt"],
            cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "kaggle_drive_backup_install.log",
        )

    _tl_split_path = REPO_DIR / "data" / "splits" / "full_metadata_with_splits.csv"
    _tl_split_sha256 = file_sha256(_tl_split_path) if _tl_split_path.exists() else None

    print(f"Working root:         {LOCAL_TRANSFER_ROOT}")
    print(f"Attached dataset root: {PERSISTENT_TRANSFER_ROOT or '(none configured)'}")
    print(f"KAGGLE_RESTORE_SOURCE={KAGGLE_RESTORE_SOURCE}  ENABLE_KAGGLE_DRIVE_BACKUP={ENABLE_KAGGLE_DRIVE_BACKUP}")
    print(f"AUTO_RESUME={AUTO_RESUME}  SKIP_COMPLETED={SKIP_COMPLETED}\n")

    if KAGGLE_RESTORE_SOURCE in ("attached_dataset", "auto") and PERSISTENT_TRANSFER_ROOT is not None:
        for _seed in TRANSFER_SEEDS:
            _mgr = PersistentArtifactManager(
                "volo_d1_face_only_pretrained", _seed, LOCAL_TRANSFER_ROOT, PERSISTENT_TRANSFER_ROOT,
                sync_after_epoch=False,
            )
            _restored = _mgr.restore_seed()
            if _restored:
                print(f"[attached_dataset] restored {len(_restored)} file(s) for seed {_seed}.")

    if KAGGLE_RESTORE_SOURCE in ("drive", "auto") and ENABLE_KAGGLE_DRIVE_BACKUP:
        from src.utils.kaggle_drive_backup import download_file, is_configured as _drive_configured
        if _drive_configured():
            _summary_target = LOCAL_TRANSFER_ROOT / "volo_d1_face_only_pretrained" / "transfer_learning_summary.zip"
            if download_file(_summary_target, drive_filename="transfer_learning_summary.zip"):
                print(f"[drive] restored lightweight summary archive to {_summary_target}")
            else:
                print(
                    "[drive] no summary archive restored (missing secrets, no prior backup, or "
                    "network unavailable) -- continuing with whatever is under /kaggle/working."
                )
        else:
            print(
                "[drive] ENABLE_KAGGLE_DRIVE_BACKUP=True but GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON/"
                "GOOGLE_DRIVE_FOLDER_ID Kaggle Secrets are not both set -- Drive backup/restore skipped, "
                "training will still save to /kaggle/working."
            )

    print("\nSeed status:")
    for _seed in TRANSFER_SEEDS:
        _status = seed_status_report(
            "volo_d1_face_only_pretrained", _seed, LOCAL_TRANSFER_ROOT, PERSISTENT_TRANSFER_ROOT,
            expected_split_sha256=_tl_split_sha256,
        )
        print("  " + format_status_line(_status))
else:
    print("Skipped (RUN_TRANSFER_LEARNING_EXTENSION=False).")


### Live Status Table (All Experiments / Seeds)

Reusable, read-only status scan -- reruns instantly at any point (before,
during, or after training) to show every experiment/seed found under
`LOCAL_TRANSFER_ROOT`: status (`COMPLETE`/`INCOMPLETE`/`NOT STARTED`),
current stage, epoch, best score, when it was last updated, and its
checkpoint path. Reads only the small `state/*.json` files
`PersistentArtifactManager` already writes after every epoch
(`src/training/persistent_artifacts.py::scan_artifact_root`) -- never loads
a full checkpoint, so it stays fast even with many seeds/experiments.


In [ ]:
# ============================================================
# Live status table -- safe to re-run any time, including while a training
# cell above is still running in another notebook session/kernel sharing
# the same LOCAL_TRANSFER_ROOT (e.g. to check progress from a second tab).
# ============================================================

from src.training.persistent_artifacts import scan_artifact_root


def show_artifact_status_table(root=None):
    import pandas as pd

    root = root if root is not None else LOCAL_TRANSFER_ROOT
    rows = scan_artifact_root(root)
    if not rows:
        print(f"No experiment/seed artifacts found yet under {root}.")
        return None
    table = pd.DataFrame(rows, columns=["experiment", "seed", "status", "stage", "epoch", "best_score", "last_update", "checkpoint"])
    table = table.sort_values(["experiment", "seed"]).reset_index(drop=True)
    display(table)
    return table


_ = show_artifact_status_table()


In [ ]:
# ============================================================
# Table B: multi-seed run (the authoritative source for the reported
# numbers) + comparison against the best from-scratch model
# (exp_d_shared_adapters_learned_balance, reused, never retrained here).
#
# --resume --skip-completed: an already-complete seed is reused (not
# retrained); an interrupted seed resumes from its latest valid checkpoint;
# a seed that never started trains from scratch. Everything is saved under
# /kaggle/working, so it becomes part of this notebook version's output
# once saved. See docs/transfer_learning.md "Kaggle recovery".
# ============================================================

if RUN_TRANSFER_LEARNING_EXTENSION:
    tl_seeds_arg = ",".join(str(s) for s in TRANSFER_SEEDS)
    tl_cmd = [
        sys.executable, "-u", str(REPO_DIR / "scripts" / "run_transfer_learning.py"),
        "--seeds", tl_seeds_arg,
        "--storage-root", str(LOCAL_TRANSFER_ROOT),
    ]
    if AUTO_RESUME:
        tl_cmd.append("--resume")
    if SKIP_COMPLETED:
        tl_cmd.append("--skip-completed")
    if PERSISTENT_TRANSFER_ROOT is not None:
        tl_cmd += ["--persistent-root", str(PERSISTENT_TRANSFER_ROOT)]

    run_command(tl_cmd, cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "transfer_learning_table_b.log")

    tl_table_b_path = REPO_DIR / "results" / "transfer_learning" / "table_b.csv"
    tl_manifest_path = REPO_DIR / "results" / "transfer_learning" / "table_b_manifest.json"
    if tl_table_b_path.exists():
        import pandas as pd
        tl_table_b = pd.read_csv(tl_table_b_path)
        display(tl_table_b)
        tl_manifest = load_json(tl_manifest_path) if tl_manifest_path.exists() else {}
        print(json.dumps(tl_manifest, indent=2))
        shutil.copy2(tl_table_b_path, RUN_DIR / "reports" / "table_b.csv")

        tl_summary_archive = LOCAL_TRANSFER_ROOT / "volo_d1_face_only_pretrained" / "transfer_learning_summary.zip"
        if ENABLE_KAGGLE_DRIVE_BACKUP and tl_summary_archive.exists():
            from src.utils.kaggle_drive_backup import upload_file as _drive_upload_file
            if _drive_upload_file(tl_summary_archive):
                print(f"Uploaded lightweight summary archive to Google Drive: {tl_summary_archive.name}")
            else:
                print("Google Drive upload of the summary archive did not complete -- artifacts remain safe under /kaggle/working.")

        print("\nSeed status (after this run):")
        for _seed in TRANSFER_SEEDS:
            _status = seed_status_report(
                "volo_d1_face_only_pretrained", _seed, LOCAL_TRANSFER_ROOT, PERSISTENT_TRANSFER_ROOT,
                expected_split_sha256=_tl_split_sha256,
            )
            print("  " + format_status_line(_status))
    else:
        print(
            "Table B was not produced -- most likely no prepared split or no "
            "'exp_d_shared_adapters_learned_balance' checkpoint was available to reuse. "
            "See the log above for details."
        )
else:
    print("Skipped (RUN_TRANSFER_LEARNING_EXTENSION=False).")


**Optional: run a single seed only** (e.g. to resume just one incomplete seed without re-checking the other two).


In [ ]:
# ============================================================
# Optional: run ONE seed only (set SINGLE_SEED in the bootstrap cell above,
# e.g. SINGLE_SEED = 123). Skip this cell to run all of TRANSFER_SEEDS via
# the Table B cell above instead.
# ============================================================

if RUN_TRANSFER_LEARNING_EXTENSION and SINGLE_SEED is not None:
    tl_single_cmd = [
        sys.executable, "-u", str(REPO_DIR / "scripts" / "run_transfer_learning.py"),
        "--seeds", str(SINGLE_SEED),
        "--storage-root", str(LOCAL_TRANSFER_ROOT),
    ]
    if AUTO_RESUME:
        tl_single_cmd.append("--resume")
    if SKIP_COMPLETED:
        tl_single_cmd.append("--skip-completed")
    if PERSISTENT_TRANSFER_ROOT is not None:
        tl_single_cmd += ["--persistent-root", str(PERSISTENT_TRANSFER_ROOT)]
    run_command(tl_single_cmd, cwd=REPO_DIR, log_path=RUN_DIR / "logs" / f"transfer_learning_seed{SINGLE_SEED}.log")
else:
    print("Skipped (RUN_TRANSFER_LEARNING_EXTENSION=False or SINGLE_SEED is None).")


In [ ]:
# ============================================================
# Explicit scientific interpretation
# ============================================================

def _parse_leading_float(value):
    """build_transfer_learning_table renders seed-sensitive metrics as
    formatted strings, e.g. '5.234 (n=1, no std)' or '5.23 +/- 0.12 (n=3)' --
    extract the leading numeric value for comparison, or None if unparseable."""
    import math
    import re

    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    match = re.match(r"[-+]?\d*\.?\d+", str(value))
    return float(match.group()) if match else None


if RUN_TRANSFER_LEARNING_EXTENSION and tl_table_b_path.exists() and len(tl_table_b) == 2:
    _volo_row = tl_table_b[tl_table_b["Model"] == "volo_d1_face_only_pretrained"].iloc[0]
    _baseline_row = tl_table_b[tl_table_b["Model"] != "volo_d1_face_only_pretrained"].iloc[0]
    _single_seed = bool(tl_manifest.get("single_seed_no_variance_estimate", True))
    print(
        f"The externally pretrained VOLO-D1 system's Age MAE was '{_volo_row['Age MAE']}' "
        f"vs. the best from-scratch system's '{_baseline_row['Age MAE']}'; "
        f"Gender F1 was '{_volo_row['Gender F1']}' vs. '{_baseline_row['Gender F1']}'.\n\n"
        "This cannot be attributed to architecture (Transformer-style vs. residual CNN) "
        "alone, because initialization, pretraining data, capacity, input resolution, "
        "and optimizer/training schedule all differ simultaneously between the two rows -- "
        "see the explanation at the top of this section. "
        + ("Table B above is a SINGLE-SEED result with no variance estimate; treat any "
           "gap between the two rows as anecdotal, not a statistically supported claim." if _single_seed else
           "Table B above aggregates >=2 seeds per row (see the +/- std columns).")
    )
    _volo_mae, _baseline_mae = _parse_leading_float(_volo_row["Age MAE"]), _parse_leading_float(_baseline_row["Age MAE"])
    _volo_f1, _baseline_f1 = _parse_leading_float(_volo_row["Gender F1"]), _parse_leading_float(_baseline_row["Gender F1"])
    _volo_worse_on_age = _volo_mae is not None and _baseline_mae is not None and _volo_mae > _baseline_mae
    _volo_worse_on_gender = _volo_f1 is not None and _baseline_f1 is not None and _volo_f1 < _baseline_f1
    if _volo_worse_on_age or _volo_worse_on_gender:
        print(
            "\nOn at least one metric, the pretrained VOLO-D1 system performed WORSE than "
            "the from-scratch baseline. This is reported as-is -- a large pretrained model "
            "can overfit a dataset of only a few thousand images, and no hyperparameter was "
            "tuned against the test set to rescue this result."
        )
else:
    print(
        "Skipped or incomplete: RUN_TRANSFER_LEARNING_EXTENSION=False, or Table B does not "
        "have both rows (baseline checkpoint and/or VOLO run unavailable). No interpretation "
        "is printed without real numbers to interpret -- see the Table B cell above for why."
    )


**Kaggle artifact archive** -- a fuller archive than the lightweight cross-platform summary above: includes best/last checkpoints (never `previous_last.pt`, a duplicate) alongside manifests/metrics/plots/configs/completion markers/Table B, excluding dataset images, cache/temp files, and anything credential-shaped.


In [ ]:
# ============================================================
# Kaggle artifact archive: /kaggle/working/agegender_transfer_learning_artifacts.zip
# Becomes a downloadable notebook output artifact once this notebook
# version is saved. Excludes dataset images, cache directories, temporary
# files, duplicate checkpoints (previous_last.pt), and anything
# credential-shaped -- see build_summary_archive()'s docstring.
# ============================================================

if RUN_TRANSFER_LEARNING_EXTENSION:
    from src.training.persistent_artifacts import build_summary_archive

    tl_kaggle_archive_path = build_summary_archive(
        LOCAL_TRANSFER_ROOT / "volo_d1_face_only_pretrained",
        Path("/kaggle/working/agegender_transfer_learning_artifacts.zip"),
        extra_files=[
            REPO_DIR / "results" / "transfer_learning" / "table_b.csv",
            REPO_DIR / "results" / "transfer_learning" / "table_b_manifest.json",
            REPO_DIR / "results" / "transfer_learning" / "seed_metrics_index.json",
        ],
        include_best_and_last_checkpoints=True,
    )
    print(f"Kaggle artifact archive: {tl_kaggle_archive_path}")

    if ENABLE_KAGGLE_DRIVE_BACKUP:
        from src.utils.kaggle_drive_backup import upload_file as _drive_upload_file
        if _drive_upload_file(tl_kaggle_archive_path):
            print("Uploaded full artifact archive to Google Drive.")
        else:
            print("Google Drive upload did not complete -- artifacts remain safe under /kaggle/working.")
else:
    print("Skipped (RUN_TRANSFER_LEARNING_EXTENSION=False).")


## 14. Final archive

Archives this session's `RUN_DIR` (configs, manifests, logs, tests,
checkpoints, metrics, plots, reports, calibration, and any
robustness/Grad-CAM/k-NN outputs -- raw dataset images live under
`REPO_DIR/data/raw`, entirely outside `RUN_DIR`, so they are never
included) directly under `/kaggle/working`, so it becomes part of this
notebook version's own output once saved. Optionally backs the archive up
to Google Drive too, under the same `{RUN_ID}_archive.zip` name a *future*
follow-up notebook's `PREVIOUS_RUN_ID`/`ENABLE_KAGGLE_DRIVE_BACKUP` restore
(section 3 above) looks for.


In [ ]:
# ============================================================
# Final archive
# Archive contains only this run's directory: configs, manifests, logs,
# tests, checkpoints, metrics, plots, reports, calibration, and any
# robustness/Grad-CAM/k-NN outputs. Raw dataset images live under
# REPO_DIR/data/raw, entirely outside RUN_DIR, so they are never included.
# ============================================================

archive_base = RUN_DIR.parent / f"{RUN_ID}_archive"
local_archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR)
print(f"Created local archive: {local_archive_path}")

kaggle_zip_path = Path("/kaggle/working") / f"AgeGender_{RUN_ID}.zip"
safe_copy2(local_archive_path, kaggle_zip_path)
print(f"Archive available for download: {kaggle_zip_path}")
print("Find it under this notebook's 'Output' tab (or Data > Output) after the run finishes, to download it.")

if ENABLE_KAGGLE_DRIVE_BACKUP:
    from src.utils.kaggle_drive_backup import upload_file as _drive_upload_file
    if _drive_upload_file(local_archive_path, drive_filename=f"{RUN_ID}_archive.zip"):
        print(f"Uploaded run archive to Google Drive as {RUN_ID}_archive.zip "
              "(a future follow-up notebook can restore it via PREVIOUS_RUN_ID/ENABLE_KAGGLE_DRIVE_BACKUP).")
    else:
        print("Google Drive upload did not complete -- the archive remains safe under /kaggle/working.")

print("\nRun complete.")
print(f"RUN_ID:  {RUN_ID}")
print(f"RUN_DIR: {RUN_DIR}")
